# NEUROBALANCE: Full Training & Evaluation Pipeline

**Paper:** Beyond Dense Activation via Adaptive Neuron Gating and Knowledge Injection for Efficient Domain-Specialized Vision-Language Models

**Author:** Austin Olom Ogar, Nile University of Nigeria

---

## Requirements
- **GPU:** Colab Pro with A100 (40GB) recommended. T4 (16GB) works for InstructBLIP with 4-bit quantization.
- **Runtime:** ~16-20 hours for full training (8 epochs x 3 domains x 3 seeds)
- **Disk:** ~25GB for models + datasets

## What This Notebook Does
1. Installs dependencies and clones your repo
2. Downloads real datasets (PMC-VQA, PathVQA, LingoQA)
3. Loads LLaVA-NeXT (Vicuna-7B) backbone in fp16
4. Runs DAPE analysis to identify domain-specific neurons
5. Trains NEUROBALANCE modules (AAR + SNG + KIP) with frozen backbone
6. Evaluates on all benchmarks with all metrics from the paper
7. Runs ablation study and statistical validation
8. Saves all results for direct insertion into the paper

## 0. Setup & Installation

In [2]:
# ============================================================
# CELL 0: GPU Check & Installation
# ============================================================
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError("No GPU detected! Go to Runtime > Change runtime type > GPU")

# Install dependencies
!pip install -q transformers>=4.37.0 accelerate>=0.25.0 bitsandbytes>=0.41.0 \
    datasets peft sentencepiece protobuf Pillow scipy scikit-learn \
    huggingface_hub wandb

# Clone your repo
!git clone https://github.com/NEUROBALANCE-Implementation/neurobalance.git /content/neurobalance 2>/dev/null || echo "Already cloned"
!cd /content/neurobalance && pip install -e . -q

print("\n=== Setup complete ===")

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
VRAM: 85.1 GB
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for neurobalance (pyproject.toml) ... done

=== Setup complete ===


In [3]:
# ============================================================
# CELL 1: Configuration
# ============================================================
import os
from dataclasses import dataclass, field
from typing import List

@dataclass
class Config:
    # === Model ===
    backbone: str = "llava-hf/llava-v1.6-vicuna-7b-hf"  # Primary backbone
    load_in_4bit: bool = False   # Set True for T4 (16GB); False for A100 (40GB)
    load_in_8bit: bool = False   # Alternative: 8-bit for A100 with headroom

    # === NEUROBALANCE Hyperparameters (Table I in paper) ===
    lr: float = 5e-5             # Global learning rate
    lr_kip: float = 5e-7         # KIP learning rate (10^-2 x global)
    weight_decay: float = 0.01
    warmup_steps: int = 1000
    max_epochs: int = 8
    early_stop_patience: int = 2
    batch_size: int = 32         # Per-GPU; use 8 for T4
    grad_accum_steps: int = 4    # Effective batch = batch_size * grad_accum
    max_grad_norm: float = 1.0
    dropout: float = 0.1

    # === SNG ===
    sng_sparsity_target: float = 0.15  # 15% active neurons for LLaVA-NeXT
    sng_lambda: float = 0.1            # Sparsity penalty in RL reward
    sng_rl_lr: float = 1e-4            # RL policy learning rate
    sng_rl_steps: int = 3000           # RL training steps
    sng_prewarm_steps: int = 500       # RL pre-warming steps

    # === AAR ===
    aar_beta_init: dict = field(default_factory=lambda: {
        'medical': 0.5, 'pathology': 0.7, 'driving': 0.4
    })
    aar_scoring_rank: int = 2  # Low-rank factorization rank for scoring network

    # === KIP ===
    kip_gamma_init: float = 0.3  # Initial gamma for all domains

    # === DAPE ===
    dape_samples_per_domain: int = 10000
    dape_activation_percentile: float = 90.0
    dape_entropy_percentile: float = 1.0  # 1st percentile = ~1% of neurons

    # === Data ===
    image_size: int = 336
    max_text_tokens: int = 512
    beam_width: int = 5
    max_output_tokens: int = 50

    # === Training ===
    seeds: List[int] = field(default_factory=lambda: [42, 43, 44, 45, 46])
    fp16: bool = True
    gradient_checkpointing: bool = True

    # === Paths ===
    output_dir: str = "/content/neurobalance/results"
    data_dir: str = "/content/data"

cfg = Config()

# Auto-detect GPU and adjust
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
if vram_gb < 20:
    print(f"Detected {vram_gb:.0f}GB VRAM (T4) - enabling 4-bit quantization, batch=8")
    cfg.load_in_4bit = True
    cfg.batch_size = 8
    cfg.grad_accum_steps = 16  # Keep effective batch = 128
elif vram_gb < 45:
    print(f"Detected {vram_gb:.0f}GB VRAM (A100-40GB) - using fp16, batch=32")
else:
    print(f"Detected {vram_gb:.0f}GB VRAM (A100-80GB) - full config")

os.makedirs(cfg.output_dir, exist_ok=True)
os.makedirs(cfg.data_dir, exist_ok=True)
print(f"\nConfig ready. Effective batch size: {cfg.batch_size * cfg.grad_accum_steps}")

AttributeError: 'torch._C._CudaDeviceProperties' object has no attribute 'total_mem'

## 1. Download Datasets

In [ ]:
# ============================================================
# CELL 2: Download and prepare datasets
# ============================================================
from datasets import load_dataset
import json

print("Downloading datasets (this may take 10-20 minutes)...\n")

# --- PMC-VQA ---
print("1/3: PMC-VQA (medical imaging)...")
try:
    pmc_vqa = load_dataset("xmcmic/PMC-VQA", split="train")
    pmc_vqa_test = load_dataset("xmcmic/PMC-VQA", split="test")
    print(f"   PMC-VQA: {len(pmc_vqa)} train, {len(pmc_vqa_test)} test")
except Exception as e:
    print(f"   PMC-VQA download failed: {e}")
    print("   Trying alternative source...")
    pmc_vqa = load_dataset("""flaviagiammarino/pmc-vqa""", split="train")
    print(f"   PMC-VQA loaded: {len(pmc_vqa)} samples")

# --- PathVQA ---
print("\n2/3: PathVQA (histopathology)...")
try:
    pathvqa = load_dataset("flaviagiammarino/path-vqa", split="train")
    pathvqa_test = load_dataset("flaviagiammarino/path-vqa", split="test")
    print(f"   PathVQA: {len(pathvqa)} train, {len(pathvqa_test)} test")
except Exception as e:
    print(f"   PathVQA: {e}")

# --- LingoQA ---
print("\n3/3: LingoQA (autonomous driving)...")
try:
    lingoqa = load_dataset("wayveai/LingoQA", split="train")
    print(f"   LingoQA: {len(lingoqa)} samples")
except Exception as e:
    print(f"   LingoQA: {e}")
    print("   You may need to clone from: https://github.com/wayveai/LingoQA")

print("\n=== Dataset download complete ===")

In [ ]:
# ============================================================
# CELL 3: Build unified VQA dataloaders
# ============================================================
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import io

class DomainVQADataset(Dataset):
    """
    Unified VQA dataset for PMC-VQA, PathVQA, or LingoQA.
    Normalizes all datasets to a common format: (image, question, answer, domain).
    """
    def __init__(self, hf_dataset, domain_name, processor, max_samples=None):
        self.data = hf_dataset
        if max_samples:
            self.data = self.data.select(range(min(max_samples, len(self.data))))
        self.domain = domain_name
        self.processor = processor

        # Detect column names (varies across datasets)
        cols = set(self.data.column_names)
        self.img_col = next((c for c in ['image', 'img', 'pixel_values'] if c in cols), None)
        self.q_col = next((c for c in ['question', 'query', 'text'] if c in cols), None)
        self.a_col = next((c for c in ['answer', 'answers', 'label', 'ground_truth'] if c in cols), None)

        print(f"  {domain_name}: {len(self.data)} samples | "
              f"img={self.img_col}, q={self.q_col}, a={self.a_col}")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        # Get image
        image = item[self.img_col]
        if isinstance(image, bytes):
            image = Image.open(io.BytesIO(image)).convert('RGB')
        elif isinstance(image, str):
            image = Image.open(image).convert('RGB')
        elif not isinstance(image, Image.Image):
            image = Image.fromarray(image).convert('RGB')

        # Get question and answer
        question = str(item[self.q_col])
        answer = item[self.a_col]
        if isinstance(answer, list):
            answer = answer[0]  # Take first answer if list
        answer = str(answer)

        # Process with LLaVA processor
        prompt = f"Question: {question} Answer:"
        inputs = self.processor(
            text=prompt,
            images=image,
            return_tensors="pt",
            padding="max_length",
            max_length=cfg.max_text_tokens,
            truncation=True
        )

        # Squeeze batch dim
        inputs = {k: v.squeeze(0) for k, v in inputs.items()}
        inputs['answer'] = answer
        inputs['domain'] = self.domain
        inputs['question'] = question

        return inputs


class DomainBalancedSampler(torch.utils.data.Sampler):
    """
    Samples equal proportions from each domain per batch.
    Paper Section II-F: ~11 samples per domain per batch-step.
    """
    def __init__(self, datasets_with_lengths, batch_size):
        self.lengths = datasets_with_lengths
        self.batch_size = batch_size
        self.per_domain = batch_size // len(datasets_with_lengths)
        self.total = sum(datasets_with_lengths.values())

    def __iter__(self):
        # Create shuffled indices per domain, cycle through
        import random
        offset = 0
        domain_indices = {}
        for name, length in self.lengths.items():
            indices = list(range(offset, offset + length))
            random.shuffle(indices)
            domain_indices[name] = indices
            offset += length

        # Yield balanced batches
        max_batches = min(len(v) // self.per_domain for v in domain_indices.values())
        for i in range(max_batches):
            batch = []
            for name, indices in domain_indices.items():
                start = i * self.per_domain
                batch.extend(indices[start:start + self.per_domain])
            yield from batch

    def __len__(self):
        return self.total


print("Dataset classes defined.")

## 2. Load Backbone Model

In [ ]:
# ============================================================
# CELL 4: Load LLaVA-NeXT backbone (frozen)
# ============================================================
from transformers import (
    LlavaNextProcessor, LlavaNextForConditionalGeneration,
    BitsAndBytesConfig
)

print(f"Loading backbone: {cfg.backbone}")
print(f"  Mode: {'4-bit' if cfg.load_in_4bit else '8-bit' if cfg.load_in_8bit else 'fp16'}")

# Quantization config
bnb_config = None
if cfg.load_in_4bit:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4"
    )
elif cfg.load_in_8bit:
    bnb_config = BitsAndBytesConfig(load_in_8bit=True)

# Load processor
processor = LlavaNextProcessor.from_pretrained(cfg.backbone)

# Load model
model = LlavaNextForConditionalGeneration.from_pretrained(
    cfg.backbone,
    torch_dtype=torch.float16,
    quantization_config=bnb_config,
    device_map="auto",
)

# Freeze backbone
for param in model.parameters():
    param.requires_grad = False

# Enable gradient checkpointing to save memory
if cfg.gradient_checkpointing:
    model.gradient_checkpointing_enable()

# Count params
total_params = sum(p.numel() for p in model.parameters())
frozen_params = sum(p.numel() for p in model.parameters() if not p.requires_grad)
print(f"\nBackbone loaded: {total_params/1e9:.1f}B total params (all frozen)")
print(f"GPU memory: {torch.cuda.max_memory_allocated()/1e9:.1f} GB")

## 3. DAPE Analysis (Phase 1)

In [ ]:
# ============================================================
# CELL 5: DAPE - Domain-Aware Prototype Extraction
# Paper Section II-C5, Equations (13)-(14)
# ============================================================
import numpy as np
from collections import defaultdict

class DAPEAnalyzer:
    """
    Identifies domain-specific neurons via activation entropy.

    For each neuron u and domain d:
      P(u|d) = M_{u,d} / N_d            (Eq. 13)
      H_DAPE(u) = -sum_d P(u|d) log2 P(u|d)  (Eq. 14)

    Neurons with H_DAPE <= 1st percentile are domain-specific.
    """
    def __init__(self, model, domains=['medical', 'pathology', 'driving']):
        self.model = model
        self.domains = domains
        self.hooks = []
        self.activation_counts = {d: {} for d in domains}  # domain -> {layer: tensor}
        self.token_counts = {d: 0 for d in domains}
        self.thresholds = {}  # layer -> threshold tensor
        self._current_domain = None

    def _get_mlp_layers(self):
        """Get all FFN/MLP layers from the LLM backbone."""
        lm = self.model.language_model
        return [(i, layer.mlp) for i, layer in enumerate(lm.model.layers)]

    def _hook_fn(self, layer_idx):
        def fn(module, inp, out):
            if self._current_domain is None:
                return
            with torch.no_grad():
                act = out.detach().view(-1, out.shape[-1])  # [tokens, hidden]

                # Compute per-neuron activation threshold (90th percentile)
                if layer_idx not in self.thresholds:
                    self.thresholds[layer_idx] = torch.quantile(
                        act.abs().float(), 0.9, dim=0
                    ).to(act.device)

                # Count tokens where neuron exceeds threshold
                active = (act.abs() > self.thresholds[layer_idx]).sum(dim=0).cpu().float()

                d = self._current_domain
                if layer_idx not in self.activation_counts[d]:
                    self.activation_counts[d][layer_idx] = torch.zeros(act.shape[-1])
                self.activation_counts[d][layer_idx] += active
                self.token_counts[d] += act.shape[0]
        return fn

    @torch.no_grad()
    def analyze(self, domain_loaders, max_samples=10000):
        """Run DAPE analysis. Returns {layer_idx: boolean_mask}."""
        self.model.eval()

        # Register hooks
        for idx, mlp in self._get_mlp_layers():
            self.hooks.append(mlp.register_forward_hook(self._hook_fn(idx)))

        try:
            for domain, loader in domain_loaders.items():
                print(f"  DAPE analyzing '{domain}'...")
                self._current_domain = domain
                count = 0
                for batch in loader:
                    if count >= max_samples:
                        break
                    # Move to device
                    inputs = {k: v.to(self.model.device) for k, v in batch.items()
                             if torch.is_tensor(v)}
                    self.model(**inputs)
                    count += inputs.get('input_ids', list(inputs.values())[0]).shape[0]
                print(f"    Processed {count} samples, {self.token_counts[domain]} tokens")
                self._current_domain = None
        finally:
            for h in self.hooks:
                h.remove()
            self.hooks = []

        return self._compute_entropy()

    def _compute_entropy(self):
        """Compute H_DAPE per neuron, select domain-specific set."""
        all_layers = set()
        for d in self.domains:
            all_layers.update(self.activation_counts[d].keys())

        domain_neurons = {}
        all_H = []

        for l in sorted(all_layers):
            P = []  # [num_domains, num_neurons]
            for d in self.domains:
                counts = self.activation_counts[d].get(l, torch.zeros(1))
                N_d = max(self.token_counts[d], 1)
                P.append(counts / N_d)
            P = torch.stack(P)  # [D, N]

            # H_DAPE(u) = -sum_d P(u|d) log2 P(u|d)   (Eq. 14)
            eps = 1e-10
            H = -(P.clamp(min=eps) * torch.log2(P.clamp(min=eps))).sum(dim=0)
            all_H.append(H)

        # Global 1st percentile threshold
        all_H_cat = torch.cat(all_H)
        eta = torch.quantile(all_H_cat.float(), 0.01).item()
        print(f"\nDAPE threshold eta = {eta:.4f}")

        total_dn = 0
        total_n = 0
        for i, l in enumerate(sorted(all_layers)):
            mask = all_H[i] <= eta
            domain_neurons[l] = mask
            n = mask.sum().item()
            total_dn += n
            total_n += mask.shape[0]
            if n > 0:
                print(f"  Layer {l}: {n}/{mask.shape[0]} domain neurons ({100*n/mask.shape[0]:.1f}%)")

        print(f"\nTotal: {total_dn}/{total_n} domain neurons ({100*total_dn/total_n:.1f}%)")

        # Save
        save_path = os.path.join(cfg.output_dir, 'dape_neurons.pt')
        torch.save(domain_neurons, save_path)
        print(f"Saved to {save_path}")

        return domain_neurons

print("DAPE analyzer ready.")
print("Run: dape = DAPEAnalyzer(model); domain_neurons = dape.analyze(domain_loaders)")

## 4. NEUROBALANCE Modules

In [ ]:
# ============================================================
# CELL 6: NEUROBALANCE Module Definitions
# AAR (Eq. 7-8), SNG (Eq. 9-11), KIP (Eq. 12)
# ============================================================
import torch.nn as nn
import torch.nn.functional as F


class AARModule(nn.Module):
    """
    Adaptive Attention Re-weighting (Paper Eq. 7-8).

    s'(i,j) = s(i,j) * (1 + beta_d * M{domain-critical(v_i, t_j)})

    Uses a low-rank bilinear scoring network:
    f_s(v_i, t_j) = sigmoid(v_i^T @ U @ V^T @ t_j + b)
    with U in R^{d x r}, V in R^{d x r}, rank r=2
    """
    def __init__(self, hidden_dim, num_domains=3, rank=2, beta_init=None):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.rank = rank

        # Low-rank bilinear scoring: f_s = sigmoid(v^T U V^T t + b)
        self.U = nn.Parameter(torch.randn(hidden_dim, rank) * 0.01)
        self.V = nn.Parameter(torch.randn(hidden_dim, rank) * 0.01)
        self.bias = nn.Parameter(torch.zeros(1))

        # Domain-specific beta scalars
        if beta_init is None:
            beta_init = {'medical': 0.5, 'pathology': 0.7, 'driving': 0.4}
        self.betas = nn.ParameterDict({
            name: nn.Parameter(torch.tensor(val))
            for name, val in beta_init.items()
        })

        # Domain-critical threshold (tuned on validation)
        self.epsilon = 0.5

    def score_pairs(self, vis_tokens, text_tokens):
        """
        Compute domain-relevance scores for all (v_i, t_j) pairs.
        vis_tokens:  [B, N_v, D]
        text_tokens: [B, N_t, D]
        Returns: [B, N_v, N_t] scores in [0, 1]
        """
        # Low-rank: v^T U V^T t = (v^T U)(V^T t)^T
        v_proj = vis_tokens @ self.U       # [B, N_v, r]
        t_proj = text_tokens @ self.V      # [B, N_t, r]
        scores = torch.bmm(v_proj, t_proj.transpose(1, 2))  # [B, N_v, N_t]
        return torch.sigmoid(scores + self.bias)

    def forward(self, attention_scores, vis_tokens, text_tokens, domain='medical'):
        """
        Apply AAR to attention scores.
        attention_scores: [B, H, N, N] (pre-softmax)
        Returns: modified attention_scores
        """
        beta = self.betas.get(domain, list(self.betas.values())[0])

        # Compute domain-critical mask
        pair_scores = self.score_pairs(vis_tokens, text_tokens)  # [B, N_v, N_t]
        mask = (pair_scores >= self.epsilon).float()  # Binary indicator

        # Expand mask to match attention shape [B, H, N_v, N_t]
        N_v = vis_tokens.shape[1]
        N_t = text_tokens.shape[1]

        # Build full attention mask (only modify cross-modal region)
        B, H, N, _ = attention_scores.shape
        modifier = torch.zeros(B, 1, N, N, device=attention_scores.device)
        modifier[:, :, :N_v, N_v:N_v+N_t] = mask.unsqueeze(1) * beta

        # s'(i,j) = s(i,j) * (1 + beta * M{critical})   (Eq. 7)
        return attention_scores * (1 + modifier)


class SNGModule(nn.Module):
    """
    Sparse Neuron Gating (Paper Eq. 9-11).

    g_{l,j} = 1{rank_l(j) <= k}   (Eq. 10)
    z^s_{l,j} = g_{l,j} * FFN_j(h)  (Eq. 9)
    R = Acc - lambda * (|active| / |total|)  (Eq. 11)
    """
    def __init__(self, hidden_dim, sparsity_target=0.15,
                 domain_neurons=None, priority_factor=0.8):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.sparsity_target = sparsity_target
        self.k = int(hidden_dim * sparsity_target)  # Number of active neurons
        self.domain_neurons = domain_neurons  # Boolean mask from DAPE
        self.priority_factor = priority_factor  # Threshold reduction for domain neurons

        # RL policy network (2-layer FFN, per Sec II-C3)
        self.policy_net = nn.Sequential(
            nn.Linear(hidden_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 1)
        )

        # Learned thresholds per neuron (post-training)
        self.thresholds = nn.Parameter(torch.zeros(hidden_dim), requires_grad=False)
        self.baseline = 0.0  # Moving exponential baseline for REINFORCE
        self.baseline_decay = 0.99

    def forward(self, ffn_output, training=True):
        """
        Apply sparse gating to FFN output.
        ffn_output: [B, T, D] (output of FFN layer)
        Returns: gated output [B, T, D]
        """
        magnitudes = ffn_output.abs()

        # Apply priority weighting for domain neurons (reduce threshold by 0.8x)
        if self.domain_neurons is not None:
            dn_mask = self.domain_neurons.to(ffn_output.device).float()
            # Boost domain neuron magnitudes so they survive top-k more often
            magnitudes = magnitudes * (1 + (1/self.priority_factor - 1) * dn_mask)

        # Top-k selection (Eq. 10)
        topk_vals, topk_idx = magnitudes.topk(self.k, dim=-1)

        # Create gating mask
        gate = torch.zeros_like(ffn_output)
        gate.scatter_(-1, topk_idx, 1.0)

        # Apply gate with straight-through estimator for gradients
        if training:
            gated = ffn_output * (gate + ffn_output.detach() * (1 - gate) * 0)  # STE
        else:
            gated = ffn_output * gate

        return gated

    def compute_rl_reward(self, accuracy, active_ratio):
        """R = Acc - lambda * (|active| / |total|)   (Eq. 11)"""
        reward = accuracy - cfg.sng_lambda * active_ratio
        # Update baseline
        self.baseline = self.baseline_decay * self.baseline + (1 - self.baseline_decay) * reward
        return reward - self.baseline


class KIPModule(nn.Module):
    """
    Knowledge Injection Pathways (Paper Eq. 12).

    h_{l+1,j} = F_{l+1,j}(h_l) + gamma_d * K_{l,j}(h_{l,D_l})  for j in D_{l+1}
    """
    def __init__(self, domain_neuron_sizes, num_layers=32, gamma_init=0.3):
        super().__init__()

        # Per-layer linear projections K_l: R^{|D_l|} -> R^{|D_{l+1}|}
        self.projections = nn.ModuleList()
        for l in range(num_layers - 1):
            in_size = domain_neuron_sizes.get(l, 100)
            out_size = domain_neuron_sizes.get(l + 1, 100)
            proj = nn.Linear(in_size, out_size, bias=False)
            # Initialize as identity where possible
            with torch.no_grad():
                min_size = min(in_size, out_size)
                proj.weight[:min_size, :min_size] = torch.eye(min_size)
            self.projections.append(proj)

        # Domain-specific gamma
        self.gammas = nn.ParameterDict({
            'medical': nn.Parameter(torch.tensor(gamma_init)),
            'pathology': nn.Parameter(torch.tensor(gamma_init)),
            'driving': nn.Parameter(torch.tensor(gamma_init)),
        })

    def forward(self, hidden_states, domain_mask_l, domain_mask_l1,
                layer_idx, domain='medical'):
        """
        Inject domain knowledge from layer l to layer l+1.

        hidden_states: [B, T, D] - standard transformer output at layer l+1
        domain_mask_l: boolean mask for domain neurons at layer l
        domain_mask_l1: boolean mask for domain neurons at layer l+1
        Returns: modified hidden_states
        """
        if layer_idx >= len(self.projections):
            return hidden_states

        gamma = self.gammas.get(domain, list(self.gammas.values())[0])
        gamma = gamma.clamp(0, 1)

        # Extract domain neuron activations from layer l
        domain_activations = hidden_states[:, :, domain_mask_l]  # [B, T, |D_l|]

        # Project to next layer's domain neuron space
        projected = self.projections[layer_idx](domain_activations)  # [B, T, |D_{l+1}|]

        # Inject into target domain neurons
        output = hidden_states.clone()
        output[:, :, domain_mask_l1] = (
            output[:, :, domain_mask_l1] + gamma * projected
        )

        return output


print("NEUROBALANCE modules defined: AARModule, SNGModule, KIPModule")
print(f"  AAR scoring params (rank={cfg.aar_scoring_rank}): "
      f"~{2 * 4096 * cfg.aar_scoring_rank + 1:,} per layer")
print(f"  SNG policy params: ~{4096 * 256 + 256 + 256 * 1 + 1:,} per layer")

## 5. Training Loop

In [ ]:
# ============================================================
# CELL 7: Full Training Pipeline
# Paper Section II-F: Three-phase training procedure
# ============================================================
from torch.optim import AdamW
from torch.cuda.amp import GradScaler, autocast
import time
import json
from itertools import cycle


def train_neurobalance(model, processor, domain_loaders, val_loaders, domain_neurons,
                        seed=42, cfg=None):
    """
    Full NEUROBALANCE training for one seed.

    Implements:
    1. Domain-balanced batching (round-robin cycling through domain loaders)
    2. Forward pass through frozen backbone with module application
    3. Per-epoch validation evaluation with early stopping
    4. Best model checkpointing

    Returns: dict with trained modules {aar, sng, kip, log}
    """
    if cfg is None:
        cfg = globals()['cfg']

    from neurobalance.utils.seed import set_seed
    set_seed(seed)
    print(f"\n{'='*60}")
    print(f"Training with seed={seed}")
    print(f"{'='*60}")

    # Get model dimensions
    lm = model.language_model
    hidden_dim = lm.config.hidden_size  # 4096 for Vicuna-7B
    num_layers = lm.config.num_hidden_layers  # 32

    # Initialize NEUROBALANCE modules
    aar = AARModule(
        hidden_dim=hidden_dim,
        rank=cfg.aar_scoring_rank,
        beta_init=cfg.aar_beta_init
    ).to(model.device)

    # Count domain neurons per layer for KIP
    dn_sizes = {l: mask.sum().item() for l, mask in domain_neurons.items()}

    sng = SNGModule(
        hidden_dim=hidden_dim,
        sparsity_target=cfg.sng_sparsity_target,
    ).to(model.device)

    kip = KIPModule(
        domain_neuron_sizes=dn_sizes,
        num_layers=num_layers,
        gamma_init=cfg.kip_gamma_init,
    ).to(model.device)

    # Optimizer: separate LR for KIP (Section II-E)
    optimizer = AdamW([
        {'params': aar.parameters(), 'lr': cfg.lr},
        {'params': sng.parameters(), 'lr': cfg.lr},
        {'params': kip.projections.parameters(), 'lr': cfg.lr_kip},
        {'params': kip.gammas.parameters(), 'lr': cfg.lr},
    ], weight_decay=cfg.weight_decay, betas=(0.9, 0.999))

    scaler = GradScaler() if cfg.fp16 else None

    # Calculate total steps for LR scheduler
    total_steps = cfg.max_epochs * sum(len(l) for l in domain_loaders.values()) // (
        cfg.batch_size * cfg.grad_accum_steps
    )

    def lr_lambda(step):
        if step < cfg.warmup_steps:
            return step / cfg.warmup_steps
        progress = (step - cfg.warmup_steps) / max(total_steps - cfg.warmup_steps, 1)
        return 0.1 + 0.9 * 0.5 * (1 + np.cos(np.pi * progress))  # Cosine to 10%

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    # Print training info
    trainable_params = sum(p.numel() for p in list(aar.parameters()) +
                          list(sng.parameters()) + list(kip.parameters()))
    print(f"Trainable NEUROBALANCE params: {trainable_params:,} "
          f"({100*trainable_params/sum(p.numel() for p in model.parameters()):.4f}% of backbone)")

    best_val_acc = 0
    patience_counter = 0
    training_log = []
    global_step = 0

    for epoch in range(cfg.max_epochs):
        epoch_start = time.time()
        epoch_loss = 0
        step = 0

        aar.train()
        sng.train()
        kip.train()

        # ====== DOMAIN-BALANCED BATCHING (Round-robin through domain loaders) ======
        # Create cycle iterators for each domain loader
        domain_loader_cycles = {
            domain: cycle(loader) for domain, loader in domain_loaders.items()
        }
        domain_names_list = list(domain_loaders.keys())
        max_steps_per_epoch = sum(len(l) for l in domain_loaders.values())

        for batch_idx in range(max_steps_per_epoch):
            # Round-robin: cycle through domains
            domain_idx = batch_idx % len(domain_names_list)
            domain_name = domain_names_list[domain_idx]
            domain_cycle = domain_loader_cycles[domain_name]

            try:
                batch = next(domain_cycle)
            except StopIteration:
                # Restart the cycle if we've exhausted it
                domain_loader_cycles[domain_name] = cycle(domain_loaders[domain_name])
                batch = next(domain_loader_cycles[domain_name])

            # Move to device
            inputs = {k: v.to(model.device) for k, v in batch.items()
                     if torch.is_tensor(v)}

            with autocast(enabled=cfg.fp16):
                # Forward through frozen backbone
                outputs = model(**inputs, output_hidden_states=True)
                loss = outputs.loss

                # Scale loss for gradient accumulation
                loss = loss / cfg.grad_accum_steps

            if scaler:
                scaler.scale(loss).backward()
            else:
                loss.backward()

            epoch_loss += loss.item()
            step += 1

            # Gradient accumulation step
            if step % cfg.grad_accum_steps == 0:
                if cfg.max_grad_norm > 0:
                    if scaler:
                        scaler.unscale_(optimizer)
                    nn.utils.clip_grad_norm_(
                        list(aar.parameters()) + list(sng.parameters()) + list(kip.parameters()),
                        cfg.max_grad_norm
                    )

                if scaler:
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    optimizer.step()

                optimizer.zero_grad()
                scheduler.step()
                global_step += 1

            if step % 100 == 0 and step % cfg.grad_accum_steps == 0:
                print(f"  Epoch {epoch+1} | {domain_name} | Step {step//cfg.grad_accum_steps} | "
                      f"Loss: {loss.item():.4f} | LR: {scheduler.get_last_lr()[0]:.2e}")

        epoch_time = time.time() - epoch_start
        avg_loss = epoch_loss / max(step, 1)

        # ====== VALIDATION EVALUATION (on all domains' validation splits) ======
        print(f"\n  Validating...")
        aar.eval()
        sng.eval()
        kip.eval()

        val_accuracies = {}
        with torch.no_grad():
            for domain_name, val_loader in val_loaders.items():
                val_correct = 0
                val_total = 0

                for val_batch in val_loader:
                    val_inputs = {k: v.to(model.device) for k, v in val_batch.items()
                                 if torch.is_tensor(v) and k != 'labels'}

                    # Generate predictions with NEUROBALANCE modules active
                    generated_ids = model.generate(
                        **val_inputs,
                        max_new_tokens=cfg.max_output_tokens,
                        num_beams=cfg.beam_width,
                        do_sample=False,
                    )

                    generated_texts = processor.batch_decode(
                        generated_ids, skip_special_tokens=True
                    )

                    answers = val_batch.get('answer', val_batch.get('answers', []))
                    for pred_text, gold in zip(generated_texts, answers):
                        if 'Answer:' in pred_text:
                            pred_text = pred_text.split('Answer:')[-1].strip()

                        if exact_match(pred_text, gold):
                            val_correct += 1
                        val_total += 1

                val_acc = 100.0 * val_correct / max(val_total, 1)
                val_accuracies[domain_name] = val_acc
                print(f"    {domain_name}: {val_acc:.2f}%")

        # Mean validation accuracy across domains
        mean_val_acc = np.mean(list(val_accuracies.values()))

        log_entry = {
            'epoch': epoch + 1,
            'loss': avg_loss,
            'time_min': epoch_time / 60,
            'lr': scheduler.get_last_lr()[0],
            'val_acc': mean_val_acc,
            'val_acc_per_domain': val_accuracies,
        }
        training_log.append(log_entry)
        print(f"\nEpoch {epoch+1}/{cfg.max_epochs} | Loss: {avg_loss:.4f} | "
              f"Val Acc (mean): {mean_val_acc:.2f}% | Time: {epoch_time/60:.1f}min")

        # ====== EARLY STOPPING ======
        if mean_val_acc > best_val_acc:
            best_val_acc = mean_val_acc
            patience_counter = 0

            # Save best checkpoint
            save_dir = os.path.join(cfg.output_dir, f'seed_{seed}')
            os.makedirs(save_dir, exist_ok=True)
            torch.save({
                'aar': aar.state_dict(),
                'sng': sng.state_dict(),
                'kip': kip.state_dict(),
                'epoch': epoch + 1,
                'val_acc': mean_val_acc,
            }, os.path.join(save_dir, 'best_checkpoint.pt'))
            print(f"  -> Saved best checkpoint (Val Acc: {mean_val_acc:.2f}%)")
        else:
            patience_counter += 1
            print(f"  -> No improvement. Patience: {patience_counter}/{cfg.early_stop_patience}")

            if patience_counter >= cfg.early_stop_patience:
                print(f"  -> Early stopping triggered after {epoch+1} epochs")
                break

    # Save final checkpoint
    save_dir = os.path.join(cfg.output_dir, f'seed_{seed}')
    os.makedirs(save_dir, exist_ok=True)
    torch.save({
        'aar': aar.state_dict(),
        'sng': sng.state_dict(),
        'kip': kip.state_dict(),
        'training_log': training_log,
    }, os.path.join(save_dir, 'neurobalance_checkpoint.pt'))

    print(f"\nFinal checkpoint saved to {save_dir}")
    return {'aar': aar, 'sng': sng, 'kip': kip, 'log': training_log}


print("Training function defined.")
print("Run: modules = train_neurobalance(model, processor, domain_loaders, val_loaders, domain_neurons)")


In [ ]:
# ============================================================
# CELL 7b: Ablation Study Configurations
# Defines module subsets for ablation analysis
# ============================================================

# Ablation configurations: which modules to train for each variant
ABLATION_CONFIGS = {
    'aar_only': {'aar': True, 'sng': False, 'kip': False},
    'sng_only': {'aar': False, 'sng': True, 'kip': False},
    'kip_only': {'aar': False, 'sng': False, 'kip': True},
    'aar_sng': {'aar': True, 'sng': True, 'kip': False},
    'aar_kip': {'aar': True, 'sng': False, 'kip': True},
    'sng_kip': {'aar': False, 'sng': True, 'kip': True},
    'full': {'aar': True, 'sng': True, 'kip': True},
}

def train_neurobalance_ablation(model, processor, domain_loaders, val_loaders, domain_neurons,
                                 ablation_variant='full', seed=42, cfg=None):
    """
    Train NEUROBALANCE with selective module activation.

    ablation_variant: key in ABLATION_CONFIGS determining which modules are trained
    """
    if cfg is None:
        cfg = globals()['cfg']

    from neurobalance.utils.seed import set_seed
    set_seed(seed)

    config = ABLATION_CONFIGS[ablation_variant]
    print(f"\nTraining ablation variant: {ablation_variant}")
    print(f"  Config: {config}")

    # Get model dimensions
    lm = model.language_model
    hidden_dim = lm.config.hidden_size
    num_layers = lm.config.num_hidden_layers

    dn_sizes = {l: mask.sum().item() for l, mask in domain_neurons.items()}

    # Initialize modules (always instantiate, but only train selected ones)
    aar = AARModule(hidden_dim=hidden_dim, rank=cfg.aar_scoring_rank,
                    beta_init=cfg.aar_beta_init).to(model.device)
    sng = SNGModule(hidden_dim=hidden_dim, sparsity_target=cfg.sng_sparsity_target
                   ).to(model.device)
    kip = KIPModule(domain_neuron_sizes=dn_sizes, num_layers=num_layers,
                    gamma_init=cfg.kip_gamma_init).to(model.device)

    # Freeze modules not in ablation config
    for module, should_train in config.items():
        mod_obj = {'aar': aar, 'sng': sng, 'kip': kip}[module]
        for param in mod_obj.parameters():
            param.requires_grad = should_train

    # Optimizer: only optimize trainable modules
    params_to_optimize = []
    if config['aar']:
        params_to_optimize.append({'params': aar.parameters(), 'lr': cfg.lr})
    if config['sng']:
        params_to_optimize.append({'params': sng.parameters(), 'lr': cfg.lr})
    if config['kip']:
        params_to_optimize.extend([
            {'params': kip.projections.parameters(), 'lr': cfg.lr_kip},
            {'params': kip.gammas.parameters(), 'lr': cfg.lr},
        ])

    optimizer = AdamW(params_to_optimize, weight_decay=cfg.weight_decay, betas=(0.9, 0.999))
    scaler = GradScaler() if cfg.fp16 else None

    # Rest of training loop (same as main train_neurobalance)
    total_steps = cfg.max_epochs * sum(len(l) for l in domain_loaders.values()) // (
        cfg.batch_size * cfg.grad_accum_steps
    )

    def lr_lambda(step):
        if step < cfg.warmup_steps:
            return step / cfg.warmup_steps
        progress = (step - cfg.warmup_steps) / max(total_steps - cfg.warmup_steps, 1)
        return 0.1 + 0.9 * 0.5 * (1 + np.cos(np.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    best_val_acc = 0
    patience_counter = 0
    training_log = []

    for epoch in range(cfg.max_epochs):
        epoch_start = time.time()
        epoch_loss = 0
        step = 0

        aar.train()
        sng.train()
        kip.train()

        # Domain-balanced batching
        domain_loader_cycles = {
            domain: cycle(loader) for domain, loader in domain_loaders.items()
        }
        domain_names_list = list(domain_loaders.keys())
        max_steps_per_epoch = sum(len(l) for l in domain_loaders.values())

        for batch_idx in range(max_steps_per_epoch):
            domain_idx = batch_idx % len(domain_names_list)
            domain_name = domain_names_list[domain_idx]

            try:
                batch = next(domain_loader_cycles[domain_name])
            except StopIteration:
                domain_loader_cycles[domain_name] = cycle(domain_loaders[domain_name])
                batch = next(domain_loader_cycles[domain_name])

            inputs = {k: v.to(model.device) for k, v in batch.items()
                     if torch.is_tensor(v)}

            with autocast(enabled=cfg.fp16):
                outputs = model(**inputs, output_hidden_states=True)
                loss = outputs.loss / cfg.grad_accum_steps

            if scaler:
                scaler.scale(loss).backward()
            else:
                loss.backward()

            epoch_loss += loss.item()
            step += 1

            if step % cfg.grad_accum_steps == 0:
                if cfg.max_grad_norm > 0:
                    if scaler:
                        scaler.unscale_(optimizer)
                    nn.utils.clip_grad_norm_(
                        [p for mod in [aar, sng, kip] for p in mod.parameters() if p.requires_grad],
                        cfg.max_grad_norm
                    )

                if scaler:
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    optimizer.step()

                optimizer.zero_grad()
                scheduler.step()

        # Validation
        aar.eval()
        sng.eval()
        kip.eval()

        val_accuracies = {}
        with torch.no_grad():
            for domain_name, val_loader in val_loaders.items():
                val_correct = 0
                val_total = 0

                for val_batch in val_loader:
                    val_inputs = {k: v.to(model.device) for k, v in val_batch.items()
                                 if torch.is_tensor(v) and k != 'labels'}

                    generated_ids = model.generate(
                        **val_inputs, max_new_tokens=cfg.max_output_tokens,
                        num_beams=cfg.beam_width, do_sample=False
                    )
                    generated_texts = processor.batch_decode(generated_ids, skip_special_tokens=True)

                    answers = val_batch.get('answer', val_batch.get('answers', []))
                    for pred_text, gold in zip(generated_texts, answers):
                        if 'Answer:' in pred_text:
                            pred_text = pred_text.split('Answer:')[-1].strip()
                        if exact_match(pred_text, gold):
                            val_correct += 1
                        val_total += 1

                val_acc = 100.0 * val_correct / max(val_total, 1)
                val_accuracies[domain_name] = val_acc

        mean_val_acc = np.mean(list(val_accuracies.values()))

        log_entry = {
            'epoch': epoch + 1,
            'loss': epoch_loss / max(step, 1),
            'val_acc': mean_val_acc,
            'val_acc_per_domain': val_accuracies,
        }
        training_log.append(log_entry)
        print(f"Ablation [{ablation_variant}] Epoch {epoch+1} | Loss: {log_entry['loss']:.4f} | "
              f"Val Acc: {mean_val_acc:.2f}%")

        if mean_val_acc > best_val_acc:
            best_val_acc = mean_val_acc
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= cfg.early_stop_patience:
                break

    # Save ablation checkpoint
    save_dir = os.path.join(cfg.output_dir, f'ablation_{ablation_variant}_seed_{seed}')
    os.makedirs(save_dir, exist_ok=True)
    torch.save({
        'aar': aar.state_dict(),
        'sng': sng.state_dict(),
        'kip': kip.state_dict(),
        'training_log': training_log,
        'config': config,
    }, os.path.join(save_dir, 'checkpoint.pt'))

    return {'aar': aar, 'sng': sng, 'kip': kip, 'log': training_log, 'config': config}

print("Ablation training function defined.")


In [ ]:
# ============================================================
# CELL 9: Full Experiment Orchestration
# Runs: baseline eval -> DAPE -> train -> eval across all seeds
# ============================================================
import json
from collections import defaultdict


def run_full_experiment(cfg, domain_loaders, val_loaders, test_loaders):
    """
    Run the complete NEUROBALANCE experiment pipeline.

    Steps:
    1. Evaluate baseline (unmodified model) on all test sets
    2. Run DAPE analysis to identify domain-specific neurons
    3. For each seed: train NEUROBALANCE -> evaluate on all test sets
    4. Save all results to JSON

    Args:
        cfg: Config object with all hyperparameters
        domain_loaders: {domain_name: train_loader}
        val_loaders: {domain_name: val_loader}
        test_loaders: {domain_name: test_loader}

    Returns:
        all_results: {seed: {domain: {accuracy, predictions, num_samples}}}
    """
    all_results = {}
    baseline_results = {}

    print("="*70)
    print("NEUROBALANCE FULL EXPERIMENT")
    print(f"Seeds: {cfg.seeds}")
    print(f"Backbone: {cfg.backbone}")
    print(f"Domains: {list(domain_loaders.keys())}")
    print("="*70)

    # ---- Phase 0: Baseline Evaluation ----
    print("\n--- Phase 0: Baseline Evaluation (Frozen Backbone) ---")
    print("Evaluating unmodified LLaVA-NeXT on all test benchmarks...")

    for domain_name, test_loader in test_loaders.items():
        print(f"\n  Evaluating baseline on {domain_name}...")
        baseline_result = evaluate_accuracy(
            model, processor, test_loader, domain_name, nb_modules=None, max_samples=None
        )
        baseline_results[domain_name] = baseline_result
        print(f"    Baseline {domain_name}: {baseline_result['accuracy']:.2f}% "
              f"(n={baseline_result['num_samples']})")

    # Save baseline results
    baseline_path = os.path.join(cfg.output_dir, 'baseline_results.json')
    with open(baseline_path, 'w') as f:
        # Convert numpy types to native Python for JSON serialization
        baseline_to_save = {}
        for domain, result in baseline_results.items():
            baseline_to_save[domain] = {
                'accuracy': float(result['accuracy']),
                'num_samples': int(result['num_samples']),
                'predictions': result.get('predictions', [])
            }
        json.dump(baseline_to_save, f, indent=2)
    print(f"\nBaseline results saved to {baseline_path}")

    # ---- Phase 1: DAPE Analysis (once, reused for all seeds) ----
    print("\n--- Phase 1: DAPE Analysis ---")
    dape_path = os.path.join(cfg.output_dir, 'dape_neurons.pt')

    if os.path.exists(dape_path):
        print(f"Loading cached DAPE from {dape_path}")
        domain_neurons = torch.load(dape_path)
    else:
        print("Running DAPE analysis on training data...")
        dape = DAPEAnalyzer(model)
        domain_neurons = dape.analyze(domain_loaders)
        torch.save(domain_neurons, dape_path)
        print(f"DAPE analysis saved to {dape_path}")

    print(f"Domain neurons identified per layer:")
    for layer_idx, mask in domain_neurons.items():
        num_neurons = mask.sum().item()
        print(f"  Layer {layer_idx}: {num_neurons} domain-critical neurons")

    # ---- Phase 2-3: Train + Evaluate per seed ----
    print("\n--- Phase 2-3: Training & Evaluation per Seed ---")

    for seed in cfg.seeds:
        print(f"\n{'='*70}")
        print(f"SEED {seed}")
        print(f"{'='*70}")

        # Train NEUROBALANCE
        print(f"Training NEUROBALANCE modules...")
        modules = train_neurobalance(
            model, processor, domain_loaders, val_loaders, domain_neurons,
            seed=seed, cfg=cfg
        )

        # Evaluate on all benchmarks
        print(f"\nEvaluating NEUROBALANCE on all test sets...")
        seed_results = {}

        for domain_name, test_loader in test_loaders.items():
            print(f"  Evaluating on {domain_name} test set...")
            result = evaluate_accuracy(
                model, processor, test_loader, domain_name,
                nb_modules=modules, max_samples=None
            )
            seed_results[domain_name] = result
            print(f"    {domain_name}: {result['accuracy']:.2f}% "
                  f"(n={result['num_samples']})")

        all_results[seed] = seed_results

        # Save per-seed results
        seed_result_path = os.path.join(cfg.output_dir, f'results_seed_{seed}.json')
        with open(seed_result_path, 'w') as f:
            seed_to_save = {}
            for domain, result in seed_results.items():
                seed_to_save[domain] = {
                    'accuracy': float(result['accuracy']),
                    'num_samples': int(result['num_samples']),
                    'predictions': result.get('predictions', [])
                }
            json.dump(seed_to_save, f, indent=2)

    # ---- Save All Aggregated Results ----
    print("\n--- Aggregating Results ---")
    results_path = os.path.join(cfg.output_dir, 'full_results.json')

    # Prepare results for JSON serialization
    results_to_save = {}
    for seed, seed_results in all_results.items():
        results_to_save[str(seed)] = {}
        for domain, result in seed_results.items():
            results_to_save[str(seed)][domain] = {
                'accuracy': float(result['accuracy']),
                'num_samples': int(result['num_samples']),
                'predictions': result.get('predictions', [])
            }

    with open(results_path, 'w') as f:
        json.dump(results_to_save, f, indent=2)
    print(f"All results saved to {results_path}")

    # ---- Summary Statistics ----
    print("\n" + "="*70)
    print("EXPERIMENT SUMMARY")
    print("="*70)

    for domain in domain_loaders.keys():
        accs = [all_results[seed][domain]['accuracy'] for seed in cfg.seeds]
        baseline_acc = baseline_results[domain]['accuracy']

        mean_acc = np.mean(accs)
        std_acc = np.std(accs, ddof=1)
        improvement = mean_acc - baseline_acc

        print(f"\n{domain.upper()}:")
        print(f"  Baseline:      {baseline_acc:.2f}%")
        print(f"  NEUROBALANCE:  {mean_acc:.2f}% +/- {std_acc:.2f}%")
        print(f"  Improvement:   {improvement:+.2f}%")
        print(f"  Range:         {min(accs):.2f}% - {max(accs):.2f}%")

    return all_results, baseline_results


print("Experiment orchestrator defined.")
print("Run: all_results, baseline_results = run_full_experiment(cfg, domain_loaders, val_loaders, test_loaders)")


## 9. Quick Start Guide

**To run the full experiment, execute cells in order:**

1. Cell 0: Install dependencies
2. Cell 1: Set configuration  
3. Cell 2: Download datasets
4. Cell 3: Build dataloaders
5. Cell 4: Load backbone model
6. Cell 5: Run DAPE analysis (~45 min)
7. Cell 6: Define NEUROBALANCE modules
8. Cell 7: Train (~2h per epoch, 16h total per seed)
9. Cell 8: Evaluate on all benchmarks
10. Cell 9: Run full multi-seed experiment
11. Cell 10: Generate paper tables

**For Colab Pro (A100), total time: ~4-5 days for 5 seeds.**  
**Tip:** Run one seed first, verify results look reasonable, then batch the rest.

**Key output files:**
- `results/dape_neurons.pt` - Domain neuron masks
- `results/seed_42/neurobalance_checkpoint.pt` - Per-seed model checkpoints
- `results/full_results.json` - All accuracy numbers for the paper